# Scratch code for scoring 

We start with loading data via O-NET and looking at how some SOCs are structured, and then come up with scoring mechanisms for the same.

In [ ]:
#|hide
import nblite; from nbdev.showdoc import show_doc; nblite.nbl_export()

/Users/bhargav/adu_dev/aisi-economy-index/.venv/lib/python3.11/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


In [ ]:
import aisi_economy_index as proj
from aisi_economy_index import const

In [ ]:
import json
import asyncio
from typing import Optional, Tuple, Dict, Any, List
import adulib.llm as llm
import inspect

In [ ]:
from dotenv import load_dotenv

load_dotenv() 

True

In [ ]:
from pydantic import BaseModel, ValidationError, conint, confloat


In [ ]:
from pathlib import Path


In [ ]:
BASE_DIR = Path("/Users/bhargav/adu_dev/aisi-economy-index/aisi_economy_index/store/data/db_30_0_excel")
OUT_DIR = Path(str(const.data_path)) / "eval_dfs"
OUT_DIR.mkdir(exist_ok=True)

## Exploring O-NET by SOC code

In [ ]:
import os
import pandas as pd

EVAL_DF_DIR = const.data_path / "eval_dfs"

def _load_csv(path):
    if not os.path.exists(path):
        # Return empty DF with no error if a file is missing
        return pd.DataFrame()
    return pd.read_csv(path)

# Load the new eval tables
skills_eval     = _load_csv(f"{EVAL_DF_DIR}/skills_eval.csv")
abilities_eval  = _load_csv(f"{EVAL_DF_DIR}/abilities_eval.csv")
knowledge_eval  = _load_csv(f"{EVAL_DF_DIR}/knowledge_eval.csv")
gwas_eval       = _load_csv(f"{EVAL_DF_DIR}/gwas_eval.csv")      # was activities_eval
dwas_eval       = _load_csv(f"{EVAL_DF_DIR}/dwas_eval.csv")      # new
tasks_eval      = _load_csv(f"{EVAL_DF_DIR}/tasks_eval.csv")

# Bundle for pipelines (with a compat alias for activities_eval → gwas_eval)
eval_dfs = dict(
    skills_eval=skills_eval,
    abilities_eval=abilities_eval,
    knowledge_eval=knowledge_eval,
    gwas_eval=gwas_eval,
    dwas_eval=dwas_eval,
    tasks_eval=tasks_eval,

    # Backwards-compat key (some old code may still look for this)
    activities_eval=gwas_eval,
)


In [ ]:
# ===========================================================================
# LOAD WORK CONTEXT AND ADD TO eval_dfs
# ===========================================================================

work_context_path = BASE_DIR / "Work Context.xlsx"
work_context_raw = pd.read_excel(work_context_path)

# Standardize columns
work_context_raw = work_context_raw.rename(columns={
    "O*NET-SOC Code": "OnetSocCode",
    "Element ID": "ElementID",
    "Element Name": "ElementName",
    "Scale ID": "ScaleID",
    "Data Value": "DataValue",
})

# Filter to CX scale (Context scale: 1-5)
work_context_eval = work_context_raw[work_context_raw["ScaleID"] == "CX"].copy()

# Add standardized columns for compatibility with helper functions
work_context_eval["context_element"] = work_context_eval["ElementName"]
work_context_eval["context_value"] = work_context_eval["DataValue"]

# Add to eval_dfs
eval_dfs["work_context_eval"] = work_context_eval

print(f"Loaded Work Context: {len(work_context_eval):,} records")
print(f"  Unique occupations: {work_context_eval['OnetSocCode'].nunique():,}")
print(f"  Unique elements: {work_context_eval['ElementID'].nunique()}")

Loaded Work Context: 49,170 records
  Unique occupations: 894
  Unique elements: 55


## Scoring 

In [ ]:
# ===========================================================================
# EXPLORATION: Examine element profiles for diverse occupations
# ===========================================================================

EXPLORATION_SOCS = {
    # High humanness (current top scorers)
    "33-1021.00": "First-Line Supervisors of Firefighting",
    "29-1214.00": "Emergency Medicine Physicians",
    "11-9171.00": "Funeral Home Managers",
    
    # Low humanness but intuitively "human" (current failures)
    "27-3043.05": "Poets, Lyricists and Creative Writers",
    "27-1013.00": "Fine Artists",
    "27-2031.00": "Dancers",
    "27-2011.00": "Actors",
    "27-2042.00": "Musicians and Singers",
    
    # Should be high physical
    "29-1141.00": "Registered Nurses",
    "33-2011.00": "Firefighters",
    "31-2021.00": "Physical Therapist Assistants",
    "27-2021.00": "Athletes and Sports Competitors",
    
    # Should be high emotional
    "21-1014.00": "Mental Health Counselors",
    "21-1021.00": "Child, Family, Social Workers",
    "19-3031.00": "Clinical Psychologists",
    "21-1011.00": "Substance Abuse Counselors",
    
    # Edge cases
    "15-1252.00": "Software Developers",
    "43-9021.00": "Data Entry Keyers",
    "53-7062.00": "Laborers and Freight Movers",
}

def explore_soc_elements(soc_code: str, eval_dfs: dict):
    """
    Print ALL elements for a SOC across skills, GWAs, and work context.
    Focus on identifying humanness-relevant elements.
    """
    print(f"\n{'='*80}")
    print(f"SOC: {soc_code}")
    print(f"{'='*80}")
    
    # --- SKILLS ---
    df = eval_dfs.get("skills_eval", pd.DataFrame())
    if not df.empty:
        sub = df[df["OnetSocCode"] == soc_code].copy()
        if not sub.empty:
            title = sub["Title"].iloc[0] if "Title" in sub.columns else "Unknown"
            print(f"Title: {title}")
            print(f"\nSKILLS ({len(sub)} elements):")
            print("-" * 60)
            
            # Sort by importance descending
            if "importance" in sub.columns:
                sub = sub.sort_values("importance", ascending=False)
            
            for _, row in sub.iterrows():
                eid = row.get("ElementID", "")
                name = row.get("skill", row.get("Element Name", ""))
                imp = row.get("importance", 0)
                lvl = row.get("level", 0)
                print(f"  {eid:<12} {name:<45} imp={imp:.2f}, lvl={lvl:.2f}")
    
    # --- GWAs ---
    df = eval_dfs.get("gwas_eval", pd.DataFrame())
    if not df.empty:
        sub = df[df["OnetSocCode"] == soc_code].copy()
        if not sub.empty:
            print(f"\nGENERALIZED WORK ACTIVITIES ({len(sub)} elements):")
            print("-" * 60)
            
            if "importance" in sub.columns:
                sub = sub.sort_values("importance", ascending=False)
            
            for _, row in sub.iterrows():
                eid = row.get("ElementID", "")
                name = row.get("activity", row.get("Element Name", ""))
                imp = row.get("importance", 0)
                lvl = row.get("level", 0)
                print(f"  {eid:<12} {name:<45} imp={imp:.2f}, lvl={lvl:.2f}")
    
    # --- WORK CONTEXT ---
    df = eval_dfs.get("work_context_eval", pd.DataFrame())
    if not df.empty:
        sub = df[df["OnetSocCode"] == soc_code].copy()
        if not sub.empty:
            print(f"\nWORK CONTEXT ({len(sub)} elements):")
            print("-" * 60)
            
            if "context_value" in sub.columns:
                sub = sub.sort_values("context_value", ascending=False)
            
            for _, row in sub.iterrows():
                eid = row.get("ElementID", "")
                name = row.get("context_element", row.get("ElementName", ""))
                val = row.get("context_value", row.get("DataValue", 0))
                print(f"  {eid:<16} {name:<50} value={val:.2f}")

# Run exploration
for soc_code, title in EXPLORATION_SOCS.items():
    explore_soc_elements(soc_code, eval_dfs)


SOC: 33-1021.00
Title: First-Line Supervisors of Firefighting and Prevention Workers

SKILLS (35 elements):
------------------------------------------------------------
  2.A.1.b      Active Listening                              imp=4.00, lvl=3.75
  2.A.2.a      Critical Thinking                             imp=3.88, lvl=3.75
  2.B.1.a      Social Perceptiveness                         imp=3.75, lvl=3.50
  2.A.1.a      Reading Comprehension                         imp=3.62, lvl=3.75
  2.A.2.d      Monitoring                                    imp=3.62, lvl=3.62
  2.B.4.e      Judgment and Decision Making                  imp=3.62, lvl=3.38
  2.B.1.f      Service Orientation                           imp=3.62, lvl=3.50
  2.B.1.b      Coordination                                  imp=3.62, lvl=3.75
  2.B.5.d      Management of Personnel Resources             imp=3.62, lvl=3.50
  2.A.1.d      Speaking                                      imp=3.62, lvl=3.62
  2.B.5.a      Time Management

In [ ]:
def catalog_all_elements(eval_dfs: dict):
    """
    Create a catalog of all unique elements across skills, GWAs, work context.
    This helps identify what's available for selection.
    """
    catalogs = {}
    
    # Skills
    df = eval_dfs.get("skills_eval", pd.DataFrame())
    if not df.empty:
        skill_elements = df[["ElementID", "skill"]].drop_duplicates()
        skill_elements = skill_elements.rename(columns={"skill": "Name"})
        catalogs["skills"] = skill_elements.sort_values("ElementID")
        print(f"Skills: {len(skill_elements)} unique elements")
    
    # GWAs
    df = eval_dfs.get("gwas_eval", pd.DataFrame())
    if not df.empty:
        gwa_elements = df[["ElementID", "activity"]].drop_duplicates()
        gwa_elements = gwa_elements.rename(columns={"activity": "Name"})
        catalogs["gwas"] = gwa_elements.sort_values("ElementID")
        print(f"GWAs: {len(gwa_elements)} unique elements")
    
    # Work Context
    df = eval_dfs.get("work_context_eval", pd.DataFrame())
    if not df.empty:
        wc_elements = df[["ElementID", "context_element"]].drop_duplicates()
        wc_elements = wc_elements.rename(columns={"context_element": "Name"})
        catalogs["work_context"] = wc_elements.sort_values("ElementID")
        print(f"Work Context: {len(wc_elements)} unique elements")
    
    return catalogs

catalogs = catalog_all_elements(eval_dfs)

# Print full catalogs
for source, df in catalogs.items():
    print(f"\n{'='*60}")
    print(f"{source.upper()} ELEMENT CATALOG")
    print(f"{'='*60}")
    for _, row in df.iterrows():
        print(f"  {row['ElementID']:<16} {row['Name']}")

Skills: 35 unique elements
GWAs: 41 unique elements
Work Context: 55 unique elements

SKILLS ELEMENT CATALOG
  2.A.1.a          Reading Comprehension
  2.A.1.b          Active Listening
  2.A.1.c          Writing
  2.A.1.d          Speaking
  2.A.1.e          Mathematics
  2.A.1.f          Science
  2.A.2.a          Critical Thinking
  2.A.2.b          Active Learning
  2.A.2.c          Learning Strategies
  2.A.2.d          Monitoring
  2.B.1.a          Social Perceptiveness
  2.B.1.b          Coordination
  2.B.1.c          Persuasion
  2.B.1.d          Negotiation
  2.B.1.e          Instructing
  2.B.1.f          Service Orientation
  2.B.2.i          Complex Problem Solving
  2.B.3.a          Operations Analysis
  2.B.3.b          Technology Design
  2.B.3.c          Equipment Selection
  2.B.3.d          Installation
  2.B.3.e          Programming
  2.B.3.g          Operations Monitoring
  2.B.3.h          Operation and Control
  2.B.3.j          Equipment Maintenance
  2.B.3.k   

In [ ]:
# =============================================================================
# ELEMENT DEFINITIONS BY DIMENSION
# =============================================================================

PHYSICAL_PRESENCE = {
    'work_context': [
        '4.C.2.a.3',      # Physical Proximity
        '4.C.1.a.2.l',    # Face-to-Face Discussions
        '4.C.1.a.4',      # Contact With Others
        '4.C.2.d.1.b',    # Spend Time Standing
        '4.C.2.d.1.g',    # Spend Time Using Your Hands
        '4.C.2.d.1.h',    # Spend Time Bending or Twisting Body
        '4.C.2.d.1.f',    # Spend Time Keeping or Regaining Balance
        '4.C.2.d.1.d',    # Spend Time Walking or Running
        '4.C.2.d.1.i',    # Spend Time Making Repetitive Motions
        '4.C.2.d.1.e',    # Spend Time Kneeling, Crouching, Stooping
        '4.C.1.b.1.e',    # Work With or Contribute to a Work Group or Team
        '4.C.2.c.1.b',    # Exposed to Disease or Infections (proximity proxy)
        '4.C.2.a.1.c',    # Outdoors, Exposed to All Weather
        '4.C.1.a.2.c',    # Public Speaking (performative)
        '4.C.2.e.1.d',    # Wear Common Protective Equipment
    ],
    'gwas': [
        '4.A.3.a.1',      # Performing General Physical Activities
        '4.A.4.a.8',      # Performing for or Working Directly with the Public
        '4.A.3.a.2',      # Handling and Moving Objects
        '4.A.3.a.4',      # Operating Vehicles, Mechanized Devices, or Equipment
        '4.A.1.b.2',      # Inspecting Equipment, Structures, or Materials
        '4.A.3.a.3',      # Controlling Machines and Processes
    ],
    'skills': [
        '2.B.1.b',        # Coordination
    ]
}

EMOTIONAL_PRESENCE = {
    'work_context': [
        '4.C.1.d.2',      # Dealing With Unpleasant, Angry, or Discourteous People
        '4.C.1.d.1',      # Conflict Situations
        '4.C.1.d.3',      # Dealing with Violent or Physically Aggressive People
        '4.C.1.b.1.f',    # Deal With External Customers or the Public
        '4.C.1.c.1',      # Health and Safety of Other Workers
        '4.C.1.c.2',      # Work Outcomes and Results of Other Workers
        '4.C.3.a.1',      # Consequence of Error
        '4.C.3.a.2.a',    # Impact of Decisions on Co-workers or Company Results
        '4.C.1.b.1.g',    # Coordinate or Lead Others
        '4.C.3.d.1',      # Time Pressure (stress context)
    ],
    'gwas': [
        '4.A.4.a.5',      # Assisting and Caring for Others
        '4.A.4.a.7',      # Resolving Conflicts and Negotiating with Others
        '4.A.4.a.4',      # Establishing and Maintaining Interpersonal Relationships
        '4.A.4.b.5',      # Coaching and Developing Others
        '4.A.4.b.4',      # Guiding, Directing, and Motivating Subordinates
        '4.A.4.b.3',      # Training and Teaching Others
        '4.A.4.b.6',      # Providing Consultation and Advice to Others
        '4.A.4.a.1',      # Interpreting the Meaning of Information for Others
        '4.A.4.b.2',      # Developing and Building Teams
        '4.A.4.a.3',      # Communicating with People Outside the Organization
        '4.A.4.a.6',      # Selling or Influencing Others
        '4.A.4.a.2',      # Communicating with Supervisors, Peers, or Subordinates
    ],
    'skills': [
        '2.B.1.a',        # Social Perceptiveness
        '2.B.1.f',        # Service Orientation
        '2.B.1.d',        # Negotiation
        '2.A.1.b',        # Active Listening
        '2.A.1.d',        # Speaking
        '2.B.1.c',        # Persuasion
        '2.B.1.e',        # Instructing
        '2.B.4.e',        # Judgment and Decision Making
    ]
}

CREATIVE_EXPRESSION = {
    'work_context': [
        '4.C.3.a.4',      # Freedom to Make Decisions
        '4.C.3.b.8',      # Determine Tasks, Priorities and Goals
        '4.C.3.c.1',      # Level of Competition
    ],
    'gwas': [
        '4.A.2.b.2',      # Thinking Creatively
        # '4.A.2.b.4',      # Developing Objectives and Strategies
        # '4.A.2.a.1',      # Judging the Qualities of Objects, Services, or People
        # '4.A.4.a.1',      # Interpreting the Meaning of Information for Others
        # '4.A.2.b.6',      # Organizing, Planning, and Prioritizing Work
    ],
    'skills': [
        '2.A.1.c',        # Writing
        # '2.A.2.b',        # Active Learning
        # '2.A.2.a',        # Critical Thinking
        # '2.B.2.i',        # Complex Problem Solving
        # '2.A.1.a',        # Reading Comprehension
    ]
}

DIMENSIONS = {
    'physical': PHYSICAL_PRESENCE,
    'emotional': EMOTIONAL_PRESENCE,
    'creative': CREATIVE_EXPRESSION
}

print("Element sets defined:")
for dim, elements in DIMENSIONS.items():
    total = sum(len(v) for v in elements.values())
    print(f"  {dim}: {total} elements (WC:{len(elements['work_context'])}, GWA:{len(elements['gwas'])}, Skills:{len(elements['skills'])})")

Element sets defined:
  physical: 22 elements (WC:15, GWA:6, Skills:1)
  emotional: 30 elements (WC:10, GWA:12, Skills:8)
  creative: 5 elements (WC:3, GWA:1, Skills:1)


In [ ]:
# =============================================================================
# NORMALIZATION FUNCTIONS
# =============================================================================

import numpy as np
import pandas as pd

def norm_importance(x):
    """Normalize importance (scale 1-5) to [0, 1]."""
    if pd.isna(x):
        return np.nan
    return float(np.clip((x - 1) / 4, 0, 1))

def norm_level(x):
    """Normalize level (scale 0-7) to [0, 1]."""
    if pd.isna(x):
        return np.nan
    return float(np.clip(x / 7, 0, 1))

def norm_work_context(x):
    """Normalize work context (scale 1-5) to [0, 1]."""
    if pd.isna(x):
        return np.nan
    return float(np.clip((x - 1) / 4, 0, 1))

print("Normalization functions defined.")
print(f"  Example: importance=4.0 -> {norm_importance(4.0):.3f}")
print(f"  Example: level=5.25 -> {norm_level(5.25):.3f}")
print(f"  Example: work_context=3.5 -> {norm_work_context(3.5):.3f}")

Normalization functions defined.
  Example: importance=4.0 -> 0.750
  Example: level=5.25 -> 0.750
  Example: work_context=3.5 -> 0.625


In [ ]:
# =============================================================================
# SCORING FUNCTIONS
# =============================================================================

def score_skills(skills_df: pd.DataFrame, element_ids: list, use_level: bool = True) -> pd.DataFrame:
    """
    Compute skill scores for specified elements.
    Returns DataFrame with OnetSocCode and score.
    """
    # Filter to relevant elements
    filtered = skills_df[skills_df['ElementID'].isin(element_ids)].copy()
    
    if filtered.empty:
        return pd.DataFrame(columns=['OnetSocCode', 'score'])
    
    # Normalize
    filtered['norm_imp'] = filtered['importance'].apply(norm_importance)
    
    if use_level and 'level' in filtered.columns:
        filtered['norm_lvl'] = filtered['level'].apply(norm_level)
        filtered['element_score'] = (filtered['norm_imp'] + filtered['norm_lvl']) / 2
    else:
        filtered['element_score'] = filtered['norm_imp']
    
    # Average across elements per SOC
    scores = filtered.groupby('OnetSocCode')['element_score'].mean().reset_index()
    scores.columns = ['OnetSocCode', 'score']
    
    return scores


def score_gwas(gwas_df: pd.DataFrame, element_ids: list, use_level: bool = True) -> pd.DataFrame:
    """
    Compute GWA scores for specified elements.
    Returns DataFrame with OnetSocCode and score.
    """
    # Filter to relevant elements
    filtered = gwas_df[gwas_df['ElementID'].isin(element_ids)].copy()
    
    if filtered.empty:
        return pd.DataFrame(columns=['OnetSocCode', 'score'])
    
    # Normalize
    filtered['norm_imp'] = filtered['importance'].apply(norm_importance)
    
    if use_level and 'level' in filtered.columns:
        filtered['norm_lvl'] = filtered['level'].apply(norm_level)
        filtered['element_score'] = (filtered['norm_imp'] + filtered['norm_lvl']) / 2
    else:
        filtered['element_score'] = filtered['norm_imp']
    
    # Average across elements per SOC
    scores = filtered.groupby('OnetSocCode')['element_score'].mean().reset_index()
    scores.columns = ['OnetSocCode', 'score']
    
    return scores


def score_work_context(wc_df: pd.DataFrame, element_ids: list) -> pd.DataFrame:
    """
    Compute work context scores for specified elements.
    Returns DataFrame with OnetSocCode and score.
    """
    # Filter to relevant elements and CX scale
    filtered = wc_df[
        (wc_df['ElementID'].isin(element_ids)) & 
        (wc_df['ScaleID'] == 'CX')
    ].copy()
    
    if filtered.empty:
        return pd.DataFrame(columns=['OnetSocCode', 'score'])
    
    # Normalize
    filtered['element_score'] = filtered['DataValue'].apply(norm_work_context)
    
    # Average across elements per SOC
    scores = filtered.groupby('OnetSocCode')['element_score'].mean().reset_index()
    scores.columns = ['OnetSocCode', 'score']
    
    return scores


def compute_dimension_scores(eval_dfs: dict, dimension_elements: dict, use_level: bool = True) -> pd.DataFrame:
    """
    Compute scores for a single dimension across all sources.
    Returns DataFrame with OnetSocCode and dimension_score.
    """
    # Get all unique SOC codes from skills (should have all occupations)
    all_socs = eval_dfs['skills_eval']['OnetSocCode'].unique()
    result = pd.DataFrame({'OnetSocCode': all_socs})
    
    source_scores = []
    
    # Work Context
    if dimension_elements.get('work_context'):
        wc_scores = score_work_context(
            eval_dfs['work_context_eval'], 
            dimension_elements['work_context']
        )
        if not wc_scores.empty:
            wc_scores = wc_scores.rename(columns={'score': 'wc_score'})
            source_scores.append(wc_scores)
    
    # GWAs
    if dimension_elements.get('gwas'):
        gwa_scores = score_gwas(
            eval_dfs['gwas_eval'], 
            dimension_elements['gwas'],
            use_level=use_level
        )
        if not gwa_scores.empty:
            gwa_scores = gwa_scores.rename(columns={'score': 'gwa_score'})
            source_scores.append(gwa_scores)
    
    # Skills
    if dimension_elements.get('skills'):
        skill_scores = score_skills(
            eval_dfs['skills_eval'], 
            dimension_elements['skills'],
            use_level=use_level
        )
        if not skill_scores.empty:
            skill_scores = skill_scores.rename(columns={'score': 'skill_score'})
            source_scores.append(skill_scores)
    
    # Merge all source scores
    for scores_df in source_scores:
        result = result.merge(scores_df, on='OnetSocCode', how='left')
    
    # Compute average across sources (equal weighting)
    score_cols = [col for col in result.columns if col.endswith('_score')]
    if score_cols:
        result['dimension_score'] = result[score_cols].mean(axis=1)
    else:
        result['dimension_score'] = np.nan
    
    return result

print("Scoring functions defined.")

Scoring functions defined.


In [ ]:
# =============================================================================
# COMPUTE ALL HUMANNESS SCORES
# =============================================================================

def compute_all_humanness_scores(eval_dfs: dict) -> pd.DataFrame:
    """
    Compute all humanness dimension scores for all occupations.
    Returns DataFrame with OnetSocCode + Title + 6 score columns.
    """
    # Get SOC codes and titles
    soc_titles = eval_dfs['skills_eval'][['OnetSocCode', 'Title']].drop_duplicates()
    result = soc_titles.copy()
    
    for dim_name, dim_elements in DIMENSIONS.items():
        # Score using importance + level
        scores_imp_lvl = compute_dimension_scores(eval_dfs, dim_elements, use_level=True)
        scores_imp_lvl = scores_imp_lvl[['OnetSocCode', 'dimension_score']].rename(
            columns={'dimension_score': f'{dim_name}_imp_lvl'}
        )
        result = result.merge(scores_imp_lvl, on='OnetSocCode', how='left')
        
        # Score using importance only
        scores_imp_only = compute_dimension_scores(eval_dfs, dim_elements, use_level=False)
        scores_imp_only = scores_imp_only[['OnetSocCode', 'dimension_score']].rename(
            columns={'dimension_score': f'{dim_name}_imp_only'}
        )
        result = result.merge(scores_imp_only, on='OnetSocCode', how='left')
    
    return result

# Compute scores
print("Computing humanness scores...")
humanness_df = compute_all_humanness_scores(eval_dfs)
print(f"Computed scores for {len(humanness_df)} occupations")
print(f"\nColumns: {humanness_df.columns.tolist()}")

Computing humanness scores...
Computed scores for 894 occupations

Columns: ['OnetSocCode', 'Title', 'physical_imp_lvl', 'physical_imp_only', 'emotional_imp_lvl', 'emotional_imp_only', 'creative_imp_lvl', 'creative_imp_only']


In [ ]:
# =============================================================================
# SUMMARY STATISTICS
# =============================================================================

score_cols = [col for col in humanness_df.columns if col.endswith('_imp_lvl') or col.endswith('_imp_only')]

print("="*80)
print("DESCRIPTIVE STATISTICS")
print("="*80)
print(humanness_df[score_cols].describe().round(3).to_string())

print("\n" + "="*80)
print("CORRELATIONS (imp_lvl scores)")
print("="*80)
imp_lvl_cols = [col for col in humanness_df.columns if col.endswith('_imp_lvl')]
print(humanness_df[imp_lvl_cols].corr().round(3).to_string())

DESCRIPTIVE STATISTICS
       physical_imp_lvl  physical_imp_only  emotional_imp_lvl  emotional_imp_only  creative_imp_lvl  creative_imp_only
count           894.000            894.000            894.000             894.000           894.000            894.000
mean              0.469              0.490              0.510               0.534             0.588              0.607
std               0.088              0.094              0.085               0.089             0.111              0.116
min               0.242              0.242              0.237               0.247             0.286              0.300
25%               0.405              0.419              0.447               0.469             0.508              0.523
50%               0.476              0.499              0.510               0.534             0.594              0.612
75%               0.534              0.561              0.571               0.597             0.677              0.702
max               0.689  

## Exploring Scores

In [ ]:
# Save presence/humanness scores to SOC_codes folder
import os
os.makedirs('SOC_codes', exist_ok=True)

# Save the humanness scores by SOC
humanness_df.to_csv('SOC_codes/presence_scores_by_soc.csv', index=False)
print(f"Saved {len(humanness_df)} SOC codes to SOC_codes/presence_scores_by_soc.csv")
humanness_df.head()

Saved 894 SOC codes to SOC_codes/presence_scores_by_soc.csv


,OnetSocCode,Title,physical_imp_lvl,physical_imp_only,emotional_imp_lvl,emotional_imp_only,creative_imp_lvl,creative_imp_only
0,11-1011.00,Chief Executives,0.495143,0.519667,0.725906,0.757319,0.765238,0.823333
1,11-1011.03,Chief Sustainability Officers,0.416444,0.434500,0.617036,0.645667,0.749583,0.801667
2,11-1021.00,General and Operations Managers,0.467857,0.491389,0.639531,0.674347,0.677540,0.707778
3,11-2011.00,Advertising and Promotions Managers,0.413802,0.420667,0.560578,0.580382,0.685833,0.722500
4,11-2021.00,Marketing Managers,0.372111,0.389333,0.611838,0.638569,0.712718,0.738611


In [ ]:
# =============================================================================
# TOP AND BOTTOM OCCUPATIONS BY DIMENSION
# =============================================================================

for dim in ['physical', 'emotional', 'creative']:
    col = f'{dim}_imp_lvl'
    
    print("\n" + "="*80)
    print(f"{dim.upper()} PRESENCE (imp+lvl)")
    print("="*80)
    
    print("\nTop 15:")
    top = humanness_df.nlargest(15, col)[['Title', col]].reset_index(drop=True)
    for i, row in top.iterrows():
        print(f"  {i+1:2d}. {row[col]:.3f}  {row['Title'][:60]}")
    
    print("\nBottom 15:")
    bot = humanness_df.nsmallest(15, col)[['Title', col]].reset_index(drop=True)
    for i, row in bot.iterrows():
        print(f"  {i+1:2d}. {row[col]:.3f}  {row['Title'][:60]}")


PHYSICAL PRESENCE (imp+lvl)

Top 15:
   1. 0.689  Manufactured Building and Mobile Home Installers
   2. 0.682  Firefighters
   3. 0.679  First-Line Supervisors of Firefighting and Prevention Worker
   4. 0.653  Forest Fire Inspectors and Prevention Specialists
   5. 0.647  Wind Turbine Service Technicians
   6. 0.646  First-Line Supervisors of Landscaping, Lawn Service, and Gro
   7. 0.646  Helpers--Extraction Workers
   8. 0.646  Electricians
   9. 0.643  Structural Iron and Steel Workers
  10. 0.643  Roofers
  11. 0.641  Stonemasons
  12. 0.639  Millwrights
  13. 0.637  Electrical Power-Line Installers and Repairers
  14. 0.636  Cement Masons and Concrete Finishers
  15. 0.632  Rotary Drill Operators, Oil and Gas

Bottom 15:
   1. 0.242  Proofreaders and Copy Markers
   2. 0.250  Credit Analysts
   3. 0.262  Business Intelligence Analysts
   4. 0.264  Environmental Economists
   5. 0.267  Personal Financial Advisors
   6. 0.268  Economists
   7. 0.270  Actuaries
   8. 0.271  Financ

In [ ]:
# =============================================================================
# OCCUPATION LOOKUP FUNCTION
# =============================================================================

def lookup_occupation(query: str, df: pd.DataFrame = None) -> pd.DataFrame:
    """
    Look up occupation(s) by SOC code or title (partial match, case-insensitive).
    Returns scores and percentile ranks for all three dimensions.
    
    Usage:
        lookup_occupation("27-2031")           # by SOC code
        lookup_occupation("model")             # by title (partial match)
        lookup_occupation("cook")              # multiple matches
        lookup_occupation("poet")              # creative occupations
    """
    if df is None:
        df = humanness_df.copy()
    
    # Compute percentile ranks if not already present
    for dim in ['physical', 'emotional', 'creative']:
        col = f'{dim}_imp_lvl'
        pct_col = f'{dim}_pct'
        if pct_col not in df.columns:
            df[pct_col] = df[col].rank(pct=True) * 100
    
    # Search by SOC code or title
    query_lower = query.lower().strip()
    
    matches = df[
        (df['OnetSocCode'].str.lower().str.contains(query_lower, na=False)) |
        (df['Title'].str.lower().str.contains(query_lower, na=False))
    ].copy()
    
    if len(matches) == 0:
        print(f"No matches found for: '{query}'")
        return pd.DataFrame()
    
    # Sort by creative score (or could be configurable)
    matches = matches.sort_values('creative_imp_lvl', ascending=False)
    
    # Display results
    print(f"Found {len(matches)} match(es) for '{query}':")
    print("="*90)
    
    for _, row in matches.iterrows():
        print(f"\n{row['Title']}")
        print(f"  SOC: {row['OnetSocCode']}")
        print(f"  ┌─────────────┬─────────┬────────────┐")
        print(f"  │ Dimension   │  Score  │ Percentile │")
        print(f"  ├─────────────┼─────────┼────────────┤")
        print(f"  │ Physical    │  {row['physical_imp_lvl']:.3f}  │   {row['physical_pct']:5.1f}%   │")
        print(f"  │ Emotional   │  {row['emotional_imp_lvl']:.3f}  │   {row['emotional_pct']:5.1f}%   │")
        print(f"  │ Creative    │  {row['creative_imp_lvl']:.3f}  │   {row['creative_pct']:5.1f}%   │")
        print(f"  └─────────────┴─────────┴────────────┘")
    
    print()
    return matches[['OnetSocCode', 'Title', 
                    'physical_imp_lvl', 'physical_pct',
                    'emotional_imp_lvl', 'emotional_pct',
                    'creative_imp_lvl', 'creative_pct']]


# =============================================================================
# FIXED COMPARISON FUNCTION
# =============================================================================

def compare_occupations(queries: list, df: pd.DataFrame = None) -> pd.DataFrame:
    """
    Compare multiple occupations side by side.
    Accepts SOC codes or title searches.
    
    Usage:
        compare_occupations(["27-2011.00", "27-2031.00", "poet", "nurse"])
    """
    if df is None:
        df = humanness_df.copy()
    
    # Compute percentile ranks if not already present
    for dim in ['physical', 'emotional', 'creative']:
        col = f'{dim}_imp_lvl'
        pct_col = f'{dim}_pct'
        if pct_col not in df.columns:
            df[pct_col] = df[col].rank(pct=True) * 100
    
    results = []
    for query in queries:
        query_lower = query.lower().strip()  # FIXED: removed errant .str
        
        # Check if it looks like a SOC code (contains digits and dashes)
        if any(c.isdigit() for c in query):
            # SOC code lookup - exact or prefix match
            matches = df[df['OnetSocCode'].str.startswith(query, na=False)]
        else:
            # Title search
            matches = df[df['Title'].str.lower().str.contains(query_lower, na=False)]
        
        if len(matches) > 0:
            results.append(matches.iloc[0])
        else:
            print(f"No match found for: '{query}'")
    
    if not results:
        print("No matches found")
        return pd.DataFrame()
    
    result_df = pd.DataFrame(results)
    
    # Display comparison table
    print("="*105)
    print("OCCUPATION COMPARISON")
    print("="*105)
    print(f"\n{'Occupation':<50} {'Physical':>12} {'Emotional':>12} {'Creative':>12}")
    print(f"{'':50} {'Score/%ile':>12} {'Score/%ile':>12} {'Score/%ile':>12}")
    print("-"*105)
    
    for _, row in result_df.iterrows():
        title = row['Title'][:48]
        print(f"{title:<50} {row['physical_imp_lvl']:.2f}/{row['physical_pct']:4.0f}%  "
              f"{row['emotional_imp_lvl']:.2f}/{row['emotional_pct']:4.0f}%  "
              f"{row['creative_imp_lvl']:.2f}/{row['creative_pct']:4.0f}%")
    
    print()
    return result_df[['OnetSocCode', 'Title', 
                      'physical_imp_lvl', 'physical_pct',
                      'emotional_imp_lvl', 'emotional_pct',
                      'creative_imp_lvl', 'creative_pct']]

print("Fixed compare_occupations function.")

Fixed compare_occupations function.


In [ ]:
# Check creative occupations
lookup_occupation("poet")

Found 1 match(es) for 'poet':

Poets, Lyricists and Creative Writers
  SOC: 27-3043.05
  ┌─────────────┬─────────┬────────────┐
  │ Dimension   │  Score  │ Percentile │
  ├─────────────┼─────────┼────────────┤
  │ Physical    │  0.272  │     1.0%   │
  │ Emotional   │  0.349  │     3.2%   │
  │ Creative    │  0.899  │   100.0%   │
  └─────────────┴─────────┴────────────┘



,OnetSocCode,Title,physical_imp_lvl,physical_pct,emotional_imp_lvl,emotional_pct,creative_imp_lvl,creative_pct
355,27-3043.05,"Poets, Lyricists and Creative Writers",0.272319,1.006711,0.349379,3.243848,0.899444,100.0


In [ ]:
# =============================================================================
# FULL ELEMENT PROFILE FOR AN OCCUPATION
# =============================================================================

def profile_occupation(query: str, eval_dfs: dict = eval_dfs, top_n: int = 20) -> None:
    """
    Show ALL element scores for an occupation, sorted by value.
    Highlights which elements are in our dimension sets.
    
    Usage:
        profile_occupation("models")
        profile_occupation("poet", top_n=30)
    """
    # Find matching SOC codes
    skills_df = eval_dfs['skills_eval']
    query_lower = query.lower().strip()
    
    matches = skills_df[
        (skills_df['OnetSocCode'].str.lower().str.contains(query_lower, na=False)) |
        (skills_df['Title'].str.lower().str.contains(query_lower, na=False))
    ][['OnetSocCode', 'Title']].drop_duplicates()
    
    if len(matches) == 0:
        print(f"No matches found for: '{query}'")
        return
    
    if len(matches) > 1:
        print(f"Multiple matches found. Showing first match.")
        print(f"All matches: {matches['Title'].tolist()}\n")
    
    soc = matches.iloc[0]['OnetSocCode']
    title = matches.iloc[0]['Title']
    
    # Collect all dimension element IDs for highlighting
    physical_ids = set(PHYSICAL_PRESENCE['work_context'] + PHYSICAL_PRESENCE['gwas'] + PHYSICAL_PRESENCE['skills'])
    emotional_ids = set(EMOTIONAL_PRESENCE['work_context'] + EMOTIONAL_PRESENCE['gwas'] + EMOTIONAL_PRESENCE['skills'])
    creative_ids = set(CREATIVE_EXPRESSION['work_context'] + CREATIVE_EXPRESSION['gwas'] + CREATIVE_EXPRESSION['skills'])
    
    def get_dimension_tag(element_id):
        tags = []
        if element_id in physical_ids:
            tags.append('P')
        if element_id in emotional_ids:
            tags.append('E')
        if element_id in creative_ids:
            tags.append('C')
        return ','.join(tags) if tags else ''
    
    print("="*95)
    print(f"FULL ELEMENT PROFILE: {title}")
    print(f"SOC: {soc}")
    print("="*95)
    print("Legend: [P]=Physical, [E]=Emotional, [C]=Creative dimension")
    
    # --- WORK CONTEXT ---
    print(f"\n{'─'*95}")
    print(f"  WORK CONTEXT (Top {top_n} by score)")
    print(f"{'─'*95}")
    
    wc_df = eval_dfs['work_context_eval']
    wc_data = wc_df[
        (wc_df['OnetSocCode'] == soc) & 
        (wc_df['ScaleID'] == 'CX')
    ][['ElementID', 'ElementName', 'DataValue']].copy()
    
    if len(wc_data) > 0:
        wc_data['norm'] = wc_data['DataValue'].apply(norm_work_context)
        wc_data['dim'] = wc_data['ElementID'].apply(get_dimension_tag)
        wc_data = wc_data.sort_values('norm', ascending=False)
        
        for i, (_, row) in enumerate(wc_data.head(top_n).iterrows()):
            dim_tag = f"[{row['dim']}]" if row['dim'] else "    "
            print(f"  {i+1:2d}. {row['norm']:.3f}  {dim_tag:6s}  {row['ElementName'][:60]}")
        
        print(f"\n  ... showing {min(top_n, len(wc_data))}/{len(wc_data)} elements")
        print(f"  Overall mean: {wc_data['norm'].mean():.3f}")
    
    # --- GWAs ---
    print(f"\n{'─'*95}")
    print(f"  GENERALIZED WORK ACTIVITIES (Top {top_n} by score)")
    print(f"{'─'*95}")
    
    gwa_df = eval_dfs['gwas_eval']
    gwa_data = gwa_df[
        gwa_df['OnetSocCode'] == soc
    ][['ElementID', 'activity', 'importance', 'level']].copy()
    
    if len(gwa_data) > 0:
        gwa_data['norm_imp'] = gwa_data['importance'].apply(norm_importance)
        gwa_data['norm_lvl'] = gwa_data['level'].apply(norm_level)
        gwa_data['norm'] = (gwa_data['norm_imp'] + gwa_data['norm_lvl']) / 2
        gwa_data['dim'] = gwa_data['ElementID'].apply(get_dimension_tag)
        gwa_data = gwa_data.sort_values('norm', ascending=False)
        
        for i, (_, row) in enumerate(gwa_data.head(top_n).iterrows()):
            dim_tag = f"[{row['dim']}]" if row['dim'] else "    "
            print(f"  {i+1:2d}. {row['norm']:.3f}  {dim_tag:6s}  {row['activity'][:60]}")
        
        print(f"\n  ... showing {min(top_n, len(gwa_data))}/{len(gwa_data)} elements")
        print(f"  Overall mean: {gwa_data['norm'].mean():.3f}")
    
    # --- SKILLS ---
    print(f"\n{'─'*95}")
    print(f"  SKILLS (Top {top_n} by score)")
    print(f"{'─'*95}")
    
    skill_df = eval_dfs['skills_eval']
    skill_data = skill_df[
        skill_df['OnetSocCode'] == soc
    ][['ElementID', 'skill', 'importance', 'level']].copy()
    
    if len(skill_data) > 0:
        skill_data['norm_imp'] = skill_data['importance'].apply(norm_importance)
        skill_data['norm_lvl'] = skill_data['level'].apply(norm_level)
        skill_data['norm'] = (skill_data['norm_imp'] + skill_data['norm_lvl']) / 2
        skill_data['dim'] = skill_data['ElementID'].apply(get_dimension_tag)
        skill_data = skill_data.sort_values('norm', ascending=False)
        
        for i, (_, row) in enumerate(skill_data.head(top_n).iterrows()):
            dim_tag = f"[{row['dim']}]" if row['dim'] else "    "
            print(f"  {i+1:2d}. {row['norm']:.3f}  {dim_tag:6s}  {row['skill'][:60]}")
        
        print(f"\n  ... showing {min(top_n, len(skill_data))}/{len(skill_data)} elements")
        print(f"  Overall mean: {skill_data['norm'].mean():.3f}")
    
    # --- SUMMARY: Coverage by dimension ---
    print(f"\n{'─'*95}")
    print(f"  DIMENSION COVERAGE SUMMARY")
    print(f"{'─'*95}")
    
    for dim_name, dim_elements in DIMENSIONS.items():
        all_dim_ids = set(dim_elements['work_context'] + dim_elements['gwas'] + dim_elements['skills'])
        
        # Get scores for dimension elements
        dim_scores = []
        
        if len(wc_data) > 0:
            wc_dim = wc_data[wc_data['ElementID'].isin(dim_elements['work_context'])]['norm'].tolist()
            dim_scores.extend(wc_dim)
        
        if len(gwa_data) > 0:
            gwa_dim = gwa_data[gwa_data['ElementID'].isin(dim_elements['gwas'])]['norm'].tolist()
            dim_scores.extend(gwa_dim)
        
        if len(skill_data) > 0:
            skill_dim = skill_data[skill_data['ElementID'].isin(dim_elements['skills'])]['norm'].tolist()
            dim_scores.extend(skill_dim)
        
        if dim_scores:
            mean_score = np.mean(dim_scores)
            print(f"  {dim_name.upper():12s}: mean={mean_score:.3f}  (n={len(dim_scores)}/{len(all_dim_ids)} elements)")

print("Profile function defined. Usage:")
print("  profile_occupation('models')")
print("  profile_occupation('poet')")
print("  profile_occupation('dancer', top_n=25)")

Profile function defined. Usage:
  profile_occupation('models')
  profile_occupation('poet')
  profile_occupation('dancer', top_n=25)


In [ ]:
compare_occupations([
    "41-9012.00",   # Models
    "27-2031.00",   # Dancers
    "27-2011.00",   # Actors
    "27-2021.00",   # Athletes and Sports Competitors
    "27-4021.00",   # Photographers
    "39-5091.00",   # Makeup Artists, Theatrical and Performance
    "27-1022.00",   # Fashion Designers
    "27-3043.05",   # Poets
    "27-1013.00",   # Fine Artists
])

OCCUPATION COMPARISON

Occupation                                             Physical    Emotional     Creative
                                                     Score/%ile   Score/%ile   Score/%ile
---------------------------------------------------------------------------------------------------------
Models                                             0.33/   7%  0.24/   0%  0.43/   8%
Dancers                                            0.51/  64%  0.41/  12%  0.53/  33%
Actors                                             0.44/  38%  0.45/  26%  0.62/  59%
Athletes and Sports Competitors                    0.60/  95%  0.53/  61%  0.61/  56%
Photographers                                      0.49/  59%  0.52/  55%  0.69/  79%
Makeup Artists, Theatrical and Performance         0.50/  59%  0.51/  49%  0.69/  79%
Fashion Designers                                  0.44/  36%  0.54/  64%  0.70/  83%
Poets, Lyricists and Creative Writers              0.27/   1%  0.35/   3%  0.90/ 100%
Fin

,OnetSocCode,Title,physical_imp_lvl,physical_pct,emotional_imp_lvl,emotional_pct,creative_imp_lvl,creative_pct
558,41-9012.00,Models,0.327437,7.046980,0.236976,0.111857,0.425754,7.718121
345,27-2031.00,Dancers,0.507266,64.429530,0.410286,12.416107,0.529782,32.774049
337,27-2011.00,Actors,0.444714,37.807606,0.450639,26.286353,0.621647,58.612975
342,27-2021.00,Athletes and Sports Competitors,0.604560,94.742729,0.533807,60.626398,0.610159,55.704698
361,27-4021.00,Photographers,0.493950,58.501119,0.521880,54.809843,0.685020,78.635347
530,39-5091.00,"Makeup Artists, Theatrical and Performance",0.495310,59.284116,0.506883,48.545861,0.686369,79.082774
331,27-1022.00,Fashion Designers,0.440397,36.129754,0.544694,64.317673,0.700476,83.109620
355,27-3043.05,"Poets, Lyricists and Creative Writers",0.272319,1.006711,0.349379,3.243848,0.899444,100.000000
328,27-1013.00,"Fine Artists, Including Painters, Sculptors, a...",0.362353,14.205817,0.303694,0.559284,0.686270,78.970917


In [ ]:
profile_occupation("51-4121")

FULL ELEMENT PROFILE: Welders, Cutters, Solderers, and Brazers
SOC: 51-4121.00
Legend: [P]=Physical, [E]=Emotional, [C]=Creative dimension

───────────────────────────────────────────────────────────────────────────────────────────────
  WORK CONTEXT (Top 20 by score)
───────────────────────────────────────────────────────────────────────────────────────────────
   1. 0.913  [P]     Wear Common Protective or Safety Equipment such as Safety Sh
   2. 0.877  [P]     Spend Time Using Your Hands to Handle, Control, or Feel Obje
   3. 0.877  [E]     Time Pressure
   4. 0.840          Importance of Being Exact or Accurate
   5. 0.837  [P]     Face-to-Face Discussions with Individuals and Within Teams
   6. 0.758          Exposed to Contaminants
   7. 0.740          Exposed to Sounds, Noise Levels that are Distracting or Unco
   8. 0.713  [P]     Work With or Contribute to a Work Group or Team
   9. 0.698  [P]     Contact With Others
  10. 0.632          Indoors, Not Environmentally Controlled

In [ ]:
profile_occupation("31-9011.00")

FULL ELEMENT PROFILE: Massage Therapists
SOC: 31-9011.00
Legend: [P]=Physical, [E]=Emotional, [C]=Creative dimension

───────────────────────────────────────────────────────────────────────────────────────────────
  WORK CONTEXT (Top 20 by score)
───────────────────────────────────────────────────────────────────────────────────────────────
   1. 0.942  [P]     Face-to-Face Discussions with Individuals and Within Teams
   2. 0.923          Indoors, Environmentally Controlled
   3. 0.905  [P]     Contact With Others
   4. 0.895  [C]     Determine Tasks, Priorities and Goals
   5. 0.885  [C]     Freedom to Make Decisions
   6. 0.837  [P]     Spend Time Making Repetitive Motions
   7. 0.830          Frequency of Decision Making
   8. 0.827  [P]     Physical Proximity
   9. 0.808  [P]     Spend Time Standing
  10. 0.808          Telephone Conversations
  11. 0.788  [E]     Deal With External Customers or the Public in General
  12. 0.740  [P]     Spend Time Bending or Twisting Your Body
  

In [ ]:
profile_occupation("35-3031.00")

FULL ELEMENT PROFILE: Waiters and Waitresses
SOC: 35-3031.00
Legend: [P]=Physical, [E]=Emotional, [C]=Creative dimension

───────────────────────────────────────────────────────────────────────────────────────────────
  WORK CONTEXT (Top 20 by score)
───────────────────────────────────────────────────────────────────────────────────────────────
   1. 0.955  [P]     Contact With Others
   2. 0.917  [P]     Spend Time Standing
   3. 0.880  [P]     Face-to-Face Discussions with Individuals and Within Teams
   4. 0.867  [P]     Spend Time Walking or Running
   5. 0.865  [P]     Work With or Contribute to a Work Group or Team
   6. 0.825          Indoors, Environmentally Controlled
   7. 0.728  [P]     Physical Proximity
   8. 0.728  [E]     Coordinate or Lead Others in Accomplishing Work Activities
   9. 0.725  [E]     Health and Safety of Other Workers
  10. 0.693  [P]     Spend Time Using Your Hands to Handle, Control, or Feel Obje
  11. 0.657  [P]     Spend Time Making Repetitive Motion

In [ ]:
profile_occupation("35-3011.00")

FULL ELEMENT PROFILE: Bartenders
SOC: 35-3011.00
Legend: [P]=Physical, [E]=Emotional, [C]=Creative dimension

───────────────────────────────────────────────────────────────────────────────────────────────
  WORK CONTEXT (Top 20 by score)
───────────────────────────────────────────────────────────────────────────────────────────────
   1. 0.925  [P]     Contact With Others
   2. 0.893  [P]     Spend Time Standing
   3. 0.812  [E]     Deal With External Customers or the Public in General
   4. 0.800  [P]     Face-to-Face Discussions with Individuals and Within Teams
   5. 0.785  [P]     Physical Proximity
   6. 0.758  [C]     Freedom to Make Decisions
   7. 0.738          Indoors, Environmentally Controlled
   8. 0.733  [E]     Dealing With Unpleasant, Angry, or Discourteous People
   9. 0.720          Importance of Being Exact or Accurate
  10. 0.682  [P]     Spend Time Making Repetitive Motions
  11. 0.675  [P]     Spend Time Walking or Running
  12. 0.672  [P]     Spend Time Using Yo

In [ ]:
profile_occupation("39-4021.00")

FULL ELEMENT PROFILE: Funeral Attendants
SOC: 39-4021.00
Legend: [P]=Physical, [E]=Emotional, [C]=Creative dimension

───────────────────────────────────────────────────────────────────────────────────────────────
  WORK CONTEXT (Top 20 by score)
───────────────────────────────────────────────────────────────────────────────────────────────
   1. 0.883  [P]     Face-to-Face Discussions with Individuals and Within Teams
   2. 0.805  [P]     Work With or Contribute to a Work Group or Team
   3. 0.802  [P]     Contact With Others
   4. 0.762          Importance of Being Exact or Accurate
   5. 0.750  [E]     Deal With External Customers or the Public in General
   6. 0.700          Indoors, Environmentally Controlled
   7. 0.688          Telephone Conversations
   8. 0.665  [E]     Time Pressure
   9. 0.650  [C]     Freedom to Make Decisions
  10. 0.647          In an Enclosed Vehicle or Operate Enclosed Equipment
  11. 0.630          Frequency of Decision Making
  12. 0.575  [P]     Spen